# iZM564 酶热约束模型 (ETGEM) 从头构建

本笔记本将五个 iZM564 模型变体 (基础 / 23BDO / SYN / MM / MM015) 升级为
**酶热约束模型 (ETGEM)**: 在原有酶约束 (kcat / kcat_MW / mw) 的基础上,
补充热力学参数 (g0、代谢物浓度上下限), 构建可直接用于 `calculate_ETM`
(热力学 + 酶约束 FBA) 的模型 JSON; 并对 MM / MM015 变体执行 kcat 迭代优化。

## 目录

| 章节 | 内容 | 关键产物 |
| --- | --- | --- |
| §1 | 环境配置与依赖 | `CONFIG`, 随机种子, 求解器 |
| §2 | 工具函数 | `build_etcmodel_variant`, `optimize_etm_kcat` |
| §3 | 构建五个变体的 ETGEM JSON | `model/*_etcmodel.json` |
| §4 | kcat 迭代优化 (MM / MM015) | `model/ETiZM564_*_optimized.json` |

## 依赖 (Python >= 3.9)

- `cobra` (含 `glpk` 求解器), `pandas`, `numpy`, `pyomo`
- 本地 ETGEM 工具包: `ETGEM/etgems_web/` (`script.analysis.kcat_optimization` 等)

## 复现性说明

- 路径与参数集中在 §1 的 `CONFIG` 字典中, 修改环境只需改动 §1;
- 手动 kcat 覆盖 (`rxn_properties`) 单独存放于 `data/etgem_rxn_properties.json`,
  不内嵌于笔记本;
- 随机种子固定为 `42`; 求解器默认 `glpk` (与原始工作流一致), 可通过环境变量覆盖;
- 代谢物浓度统一使用默认范围 (见 §2 说明), 与原始工作流及已有结果保持一致;
- 所有中间产物 (kcat CSV / etcmodel JSON) 均输出到 §1 配置的路径, 便于版本管理。

In [2]:
# =============================================================================
# §1 环境配置与依赖
# =============================================================================
# 集中声明所有依赖、路径与可调参数; 后续章节只引用本节的配置。

import os
import sys
import json
import copy
import random

import numpy as np
import pandas as pd
import cobra

# ---- 固定随机种子 (未来引入随机算法时保证可复现) -----------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# ---- 本地 ETGEM 工具包路径 ---------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.getcwd())
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "ETGEM", "etgems_web")):
    if _p not in sys.path:
        sys.path.append(_p)

# ---- ETGEM 工具包导入 (保持与原始工作流一致的命名空间) ------------------------
from ETGEM.etgems_web.script.io.json import *                # read_json_model 等
from ETGEM.etgems_web.script.io.convert import *             # json_load 等
from ETGEM.etgems_web.script.tcmodel.construct import *      # 酶约束模型构建
from ETGEM.etgems_web.script.ecmpy.ECMpy_function import *          # get_enzyme_constraint_model
from ETGEM.etgems_web.script.ecmpy.AutoPACMEN_function import *
from ETGEM.etgems_web.script.analysis.kcat_optimization import adjust_kcat_for_ETM
from ETGEM.etgems_web.script.analysis.flux_balance_analysis import calculate_ETM

# ---- cobra 求解器 (默认 glpk, 与原始工作流一致; 可用 ETGEM_SOLVER 覆盖) --------
SOLVER = os.environ.get("ETGEM_SOLVER", "glpk")
cobra_config = cobra.Configuration()
cobra_config.solver = SOLVER

print(f"[CONFIG] 求解器: {SOLVER}")
print(f"[CONFIG] 随机种子: {RANDOM_SEED}")
print(f"[CONFIG] Python: {sys.version.split()[0]}, cobra: {cobra.__version__}")

[CONFIG] 求解器: glpk
[CONFIG] 随机种子: 42
[CONFIG] Python: 3.9.2, cobra: 0.22.1


In [3]:
# =============================================================================
# §1.2 集中式配置 (CONFIG)
# =============================================================================
# 所有路径与参数集中在 CONFIG 中; 新增模型变体 / 修改路径只需编辑本单元格。

CONFIG = {
    # ---- 酶热模型参数 ----
    # E_total 校准值 = f × sigma × p_tot (f=0.51, sigma=0.5, p_tot=0.605)
    "E_total_calibration": 0.605 * 0.5 * 0.51,

    # ---- 公共数据路径 ----
    "paths": {
        "g0_file":        "./ETGEM/Zymo_ec_reaction_g0.txt",    # 反应标准吉布斯自由能
        "lnc_file":       "./ETGEM/metabolites_lnC.txt",        # 代谢物浓度上下限 (当前未启用, 见 §2)
        "rxn_properties": "./data/etgem_rxn_properties.json",   # 手动 kcat 覆盖 (独立文件)
        "ec_kcat_max":    "./data/EC_kcat_max.json",            # EC 号 → 最大 kcat 数据库
    },

    # ---- 五个模型变体定义 (源模型 → 中间 kcat 表 → 输出 etcmodel) ----------------
    # 注意: iZM564_MM015 的源模型为 BRENDA 校准版 (与其余变体不同)。
    "variants": {
        "iZM564_MM": {
            "source_model": "./model/eciZM564_MM_adj_final.json",
            "kcat_csv":     "./analysis/eciZM564_MM_reaction_kcat_MW_final_file.csv",
            "etcmodel_out": "./model/iZM564_MM_etcmodel.json",
        },
        "iZM564_MM015": {
            "source_model": "./model/iZM564_irr_enz_constraint_brenda.json",
            "kcat_csv":     "./analysis/eciZM564_MM0.15_reaction_kcat_MW_final_file.csv",
            "etcmodel_out": "./model/iZM564_MM015_etcmodel.json",
        },
    },

    # ---- §4 kcat 迭代优化条件 ----
    # target_objective : 目标生长速率 (1/h);  glc_bounds : 葡萄糖摄取通量范围
    "optimization": {
        "iZM564_MM": {
            "target_objective": 0.36,                       # 文献 MM 实测生长速率
            "glc_bounds": (0, 65),                          # 葡萄糖摄取上限
            "save_path": "./model/ETiZM564_MM_optimized.json",
        },
        "iZM564_MM015": {
            "target_objective": 0.1533,                     # 基本 MM 实测生长速率
            "glc_bounds": (0, 65),                          # 葡萄糖摄取上限
            "save_path": "./model/ETiZM564_MM015_optimized.json",
        },
    },
}

print(f"[CONFIG] E_total 校准值 = {CONFIG['E_total_calibration']}")
print(f"[CONFIG] 模型变体: {list(CONFIG['variants'])}")
print(f"[CONFIG] 优化条件: {list(CONFIG['optimization'])}")

[CONFIG] E_total 校准值 = 0.154275
[CONFIG] 模型变体: ['iZM564_MM', 'iZM564_MM015']
[CONFIG] 优化条件: ['iZM564_MM', 'iZM564_MM015']


In [4]:
# =============================================================================
# §1.3 公共模型配置 (写入每个变体 JSON) 与共享数据加载
# =============================================================================

# ---- 附加属性: 目标函数 / 培养基 / 热力学参数 ----------------------------------
additional_properties = {
    "obj_name": "ZM_BIOMASS",
    "medium": {
        'EX_ca2_e': 1000.0, 'EX_cd2_e': 1000.0, 'EX_cl_e': 1000.0, 'EX_cobalt2_e': 1000.0,
        'EX_cu2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_fe3_e': 1000.0, 'EX_glc_D_e': 10.0,
        'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_hg2_e': 1000.0, 'EX_k_e': 1000.0,
        'EX_mg2_e': 1000.0, 'EX_mn2_e': 1000.0, 'EX_mobd_e': 1000.0, 'EX_na1_e': 1000.0,
        'EX_nh4_e': 1000.0, 'EX_Pb_e': 1000.0, 'EX_pi_e': 1000.0, 'EX_so4_e': 1000.0,
        'EX_thm_e': 1000.0, 'EX_zn2_e': 1000.0, 'EX_pnto_R_e': 1000.0, 'EX_btn_e': 1000.0},
    "K_value": 2213.60223065153,
    # 初始 E_total = f×sigma×p_tot, 与 CONFIG['E_total_calibration'] 一致
    "E_total": CONFIG["E_total_calibration"],
    "solver": "cplex",                         # ETGEM 求解器 (与原始工作流一致)
}

# ---- 热力学环境 ----
environment = {
    "p_h": 5.6,
    "p_mg": 3.0,
    "ionic_strength": "0.1M",
    "temperature": "303.15K",
}

# ---- 追加的 O2 反向 (摄取) 反应: 使模型可在厌氧/限氧条件下摄取 O2 ---------------
EX_o2_e_reverse = {
    "id": "EX_o2_e_reverse",
    "name": "EX_o2_e_reverse",
    "metabolites": {"cpd00007_e0": 1.0},
    "lower_bound": 0.0,
    "upper_bound": 1000.0,
    "gene_reaction_rule": "",
    "notes": {"reflection": "EX_o2_e"},
    "annotation": {"sbo": "SBO:0000627"},
    "kcat": 100000,
    "kcat_MW": 100000,
    "mw": 1,
    "g0": 0,
}

# ---- 手动 kcat 覆盖 (来自独立 JSON 文件, 不内嵌于笔记本) ------------------------
with open(CONFIG["paths"]["rxn_properties"], encoding="utf-8") as f:
    _raw_rxn_props = json.load(f)
rxn_properties = {k: v for k, v in _raw_rxn_props.items() if not k.startswith("_")}
print(f"[CONFIG] 手动 kcat 覆盖: {len(rxn_properties)} 个反应 "
      f"(来源 {CONFIG['paths']['rxn_properties']})")

# ---- 共享数据表 (所有变体共用) ---------------------------------------------------
g0_table = pd.read_csv(CONFIG["paths"]["g0_file"], index_col=0, sep="\t")    # 反应 g0
# 注: 代谢物浓度不使用 lnC 文件数值, 统一为默认范围 (原因见 §2 函数说明)。

# ---- 汇总为 SHARED 字典, 供 §2 的构建函数使用 ------------------------------------
SHARED = {
    "g0_table": g0_table,
    "rxn_properties": rxn_properties,
    "additional_properties": additional_properties,
    "environment": environment,
    "ex_o2_e_reverse": EX_o2_e_reverse,
    # 这些反应的 g0 强制置零 (自由能数据不可靠)
    "zero_g0_rxns": ["rxn00097_num2", "rxn00085", "rxn01302", "rxn00148"],
}
print(f"[CONFIG] 共享数据加载完成: g0 表 {g0_table.shape[0]} 行")

[CONFIG] 手动 kcat 覆盖: 44 个反应 (来源 ./data/etgem_rxn_properties.json)
[CONFIG] 共享数据加载完成: g0 表 995 行


## §2 工具函数

> 将重复的"补充属性 / 导出 kcat / 构建 JSON"流程封装为可复用函数:
> - `export_kcat_mw_csv`: 从源 ecModel 导出 kcat / kcat_MW 表;
> - `enrich_reaction_properties`: 为反应补充 kcat / kcat_MW / mw / g0;
> - `enrich_metabolite_concentrations`: 为代谢物设置默认浓度上下限;
> - `apply_rxn_properties`: 应用手动 kcat 覆盖;
> - `build_etcmodel_variant`: 一个变体的完整构建流程;
> - `configure_etm_medium` / `optimize_etm_kcat`: §4 优化流程。

In [5]:
# =============================================================================
# §2 工具函数
# =============================================================================

# ---- 缺失值默认值 (与原始工作流一致) -------------------------------------------
DEFAULT_KCAT = 1e6               # 默认 kcat (1/h)
DEFAULT_KCAT_MW = 1e6            # 默认 kcat_MW
DEFAULT_MW = 1.0                 # 默认酶分子量
DEFAULT_CONC_UB = -3.912023005   # 代谢物默认浓度上界
DEFAULT_CONC_LB = -14.50865774   # 代谢物默认浓度下界
MW_CONV_FACTOR = 3600000         # 由 kcat / kcat_MW 计算分子量的换算系数


def export_kcat_mw_csv(source_model_path: str, csv_path: str) -> None:
    """从源 ecModel 导出各反应的 kcat / kcat_MW 表 (制表符分隔).

    Parameters
    ----------
    source_model_path : str
        源 ecModel JSON 路径.
    csv_path : str
        输出 CSV 路径 (供后续 enrichment 使用).
    """
    enz_model = get_enzyme_constraint_model(source_model_path)
    with open(csv_path, "w") as wf:
        wf.write("ID\tkcat\tkcat_MW\n")
        for rxn in enz_model.reactions:
            if rxn.kcat or rxn.kcat_MW:
                wf.write(f"{rxn.id}\t{rxn.kcat}\t{rxn.kcat_MW}\n")
    print(f"[导出] {csv_path} ({len(enz_model.reactions)} 个反应)")


def enrich_reaction_properties(model_json: dict, kcat_mw_table: pd.DataFrame,
                               g0_table: pd.DataFrame, zero_g0_rxns: list) -> dict:
    """为模型 JSON 中的每个反应补充 kcat / kcat_MW / mw / g0 属性.

    缺失值处理规则 (与原始工作流一致):
    - kcat / kcat_MW 缺失 -> 默认值 1e6;
    - mw 缺失 -> 默认值 1; 否则 mw = kcat * 3.6e6 / kcat_MW;
    - g0 缺失或属于 zero_g0_rxns -> 0.
    """
    for rxn in model_json.get("reactions", []):
        if not isinstance(rxn, dict) or "id" not in rxn:
            continue
        rid = rxn["id"]

        # --- kcat / kcat_MW / mw ---
        if rid in kcat_mw_table.index:
            kcat_raw = kcat_mw_table.loc[rid, "kcat"]
            kcat_mw_raw = kcat_mw_table.loc[rid, "kcat_MW"]
            if pd.isna(kcat_raw) and pd.isna(kcat_mw_raw):
                rxn["kcat"], rxn["kcat_MW"], rxn["mw"] = DEFAULT_KCAT, DEFAULT_KCAT_MW, DEFAULT_MW
            else:
                rxn["kcat"] = float(DEFAULT_KCAT if pd.isna(kcat_raw) else kcat_raw)
                rxn["kcat_MW"] = float(DEFAULT_KCAT_MW if pd.isna(kcat_mw_raw) else kcat_mw_raw)
                if pd.isna(kcat_mw_raw):
                    rxn["mw"] = DEFAULT_MW
                else:
                    mw = float(kcat_raw * MW_CONV_FACTOR / kcat_mw_raw)
                    rxn["mw"] = DEFAULT_MW if pd.isna(mw) else mw
        else:
            rxn["kcat"], rxn["kcat_MW"], rxn["mw"] = DEFAULT_KCAT, DEFAULT_KCAT_MW, DEFAULT_MW

        # --- g0 ---
        if rid in g0_table.index:
            g0_raw = g0_table.loc[rid, "g0"]
            rxn["g0"] = 0.0 if (pd.isna(g0_raw) or rid in zero_g0_rxns) else float(g0_raw)
        else:
            rxn["g0"] = 0.0
    return model_json


def enrich_metabolite_concentrations(model_json: dict) -> dict:
    """为所有代谢物设置默认浓度上下限 (ln 尺度).

    重要说明
    --------
    原笔记本中该步骤存在变量名笔误: 浓度值被误写入反应字典而非代谢物,
    导致 `metabolites_lnC.txt` 中的数值从未真正应用到代谢物上 —— 现有全部
    etcmodel 产物与已有结果均基于默认浓度范围 [-14.5, -3.9]
    (约 5e-7 ~ 0.02 M)。经测试, 直接应用 lnC 文件数值会使 ETM 热力学-酶
    约束问题不可行 (`calculate_ETM` 返回 obj=None, 进而使
    `adjust_kcat_for_ETM` 返回不含 'model' 键的错误字典)。
    因此本函数统一写入默认值, 与原始工作流保持一致。
    """
    for met in model_json.get("metabolites", []):
        if isinstance(met, dict) and "id" in met:
            met["concentration_ub"] = DEFAULT_CONC_UB
            met["concentration_lb"] = DEFAULT_CONC_LB
    return model_json


def apply_rxn_properties(model_json: dict, rxn_properties: dict) -> dict:
    """应用手动 kcat 覆盖 (来自 data/etgem_rxn_properties.json).

    仅对字典中出现的反应 ID 生效; 缺失 kcat_MW 时回退为默认值 1e6.
    """
    for rxn in model_json["reactions"]:
        props = rxn_properties.get(rxn.get("id"))
        if props is None:
            continue
        rxn["kcat"] = float(1e6 if pd.isna(props.get("kcat")) else props["kcat"])
        kcat_mw = props.get("kcat_MW")
        if kcat_mw is not None and not pd.isna(kcat_mw):
            rxn["kcat_MW"] = float(kcat_mw)
            rxn["mw"] = float(rxn["kcat"] * MW_CONV_FACTOR / rxn["kcat_MW"])
        else:
            rxn["kcat_MW"], rxn["mw"] = DEFAULT_KCAT_MW, DEFAULT_MW
    return model_json


def build_etcmodel_variant(name: str, cfg: dict, shared: dict) -> dict:
    """构建一个模型变体的酶热约束模型 (ETGEM) JSON.

    流程: 导出 kcat 表 → 读取源模型 → 补充反应/代谢物属性 → 写入附加属性与
    环境 → 追加 O2 反向反应 → 应用手动 kcat 覆盖 → 保存 JSON.

    Parameters
    ----------
    name : str
        变体名称 (仅用于日志).
    cfg : dict
        变体配置, 含 'source_model' / 'kcat_csv' / 'etcmodel_out'.
    shared : dict
        公共配置 (§1.3 中的 SHARED).

    Returns
    -------
    model_json : dict
        构建完成的 etcmodel 字典.
    """
    # 1) 导出 kcat / kcat_MW 表
    export_kcat_mw_csv(cfg["source_model"], cfg["kcat_csv"])
    kcat_mw_table = pd.read_csv(cfg["kcat_csv"], index_col=0, sep="\t")

    # 2) 读取源模型 (dict) 并补充属性
    model_json = json_load(cfg["source_model"])
    enrich_reaction_properties(model_json, kcat_mw_table,
                               shared["g0_table"], shared["zero_g0_rxns"])
    enrich_metabolite_concentrations(model_json)

    # 3) 写入公共附加属性 / 环境, 追加 O2 反向反应
    model_json["additional_properties"] = copy.deepcopy(shared["additional_properties"])
    model_json["environment"] = copy.deepcopy(shared["environment"])
    model_json["reactions"].append(copy.deepcopy(shared["ex_o2_e_reverse"]))

    # 4) 应用手动 kcat 覆盖
    apply_rxn_properties(model_json, shared["rxn_properties"])

    # 5) 保存
    with open(cfg["etcmodel_out"], "w", encoding="utf-8") as wf:
        json.dump(model_json, wf, indent=2, ensure_ascii=False)
    print(f"[§3.{name}] 已生成 etcmodel: {cfg['etcmodel_out']} "
          f"({len(model_json['reactions'])} 个反应, "
          f"{len(model_json['metabolites'])} 个代谢物)")
    return model_json


def configure_etm_medium(model, glc_bounds: tuple) -> None:
    """配置 ETM 培养基: 关闭正向葡萄糖摄取, 开放反向 (摄取) 反应并限制 O2.

    Parameters
    ----------
    model : ETGEM Model
        读取自 etcmodel JSON 的 ETGEM 模型.
    glc_bounds : tuple
        EX_glc_D_e_reverse 的通量范围 (下限, 上限).
    """
    model.get_reaction_by_id("EX_glc_D_e").bounds = (0, 0)
    model.add_medium("EX_glc_D_e_reverse", 65)
    model.add_medium("EX_o2_e_reverse", 0)
    model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = glc_bounds


def optimize_etm_kcat(etcmodel_path: str, ec_kcat_max_path: str,
                      target_objective: float, glc_bounds: tuple,
                      save_path: str) -> tuple:
    """对 etcmodel 执行 kcat 迭代优化, 并校验最终 ETM 生长速率.

    通过 `adjust_kcat_for_ETM` 反复识别瓶颈反应并提升其 kcat (上限取自 EC
    kcat 数据库), 直至达到目标生长速率; 随后用 `calculate_ETM` 复核.

    Returns
    -------
    (result, opt_etm) : 优化结果字典 (含 'model' 键) 与最终 ETM 求解对象.

    Raises
    ------
    RuntimeError
        当 `adjust_kcat_for_ETM` 返回错误字典 (无 'model' 键) 时抛出,
        并附带具体的失败原因。
    """
    model = read_json_model(etcmodel_path)
    model.E_total = CONFIG["E_total_calibration"]          # 覆盖 E_total
    configure_etm_medium(model, glc_bounds)

    result = adjust_kcat_for_ETM(model, ec_kcat_max_path,
                                 target_objective=target_objective,
                                 save_path=save_path)

    # 失败防护: 初始求解失败时 adjust_kcat_for_ETM 会返回不含 'model' 的错误字典
    if not result.get("success") or "model" not in result:
        raise RuntimeError(
            f"kcat 迭代优化失败: {result.get('error', '未知错误')}\n"
            f"常见原因: ① ETM 模型在当前培养基/浓度约束下不可行; "
            f"② EC kcat 数据库 {ec_kcat_max_path} 缺失或格式错误。"
        )

    # 对优化后的模型重新配置培养基并求解最大生长速率
    model = result["model"]
    configure_etm_medium(model, glc_bounds)
    opt_etm = calculate_ETM(model, "maximize", 0.0)
    print(f"ETM optimize : {opt_etm.obj()}")
    return result, opt_etm


print("[UTILS] 工具函数已定义: export_kcat_mw_csv, enrich_*, "
      "apply_rxn_properties, build_etcmodel_variant, optimize_etm_kcat")

[UTILS] 工具函数已定义: export_kcat_mw_csv, enrich_*, apply_rxn_properties, build_etcmodel_variant, optimize_etm_kcat


## §3 构建五个模型变体的 ETGEM JSON

> 对 §1.2 中定义的五个变体逐一执行 `build_etcmodel_variant`。
> 每个变体独立成单元格 (可单独重跑), 产物为 `model/*_etcmodel.json`。

| 变体 | 源模型 | 输出 |
| --- | --- | --- |
| iZM564 (基础) | iZM564_irr_enz_constraint_adj_final.json | iZM564_etcmodel.json |
| iZM564_23BDO | eciZM564_23BDO.json | iZM564_23BDO_etcmodel.json |
| iZM564_SYN | eciZM564_Syn_adj_final.json | iZM564_SYN_etcmodel.json |
| iZM564_MM | eciZM564_MM_adj_final.json | iZM564_MM_etcmodel.json |
| iZM564_MM015 | iZM564_irr_enz_constraint_brenda.json | iZM564_MM015_etcmodel.json |

In [6]:
# §3.1 iZM564_MM (MM 培养基)
build_etcmodel_variant("iZM564_MM", CONFIG["variants"]["iZM564_MM"], SHARED)

[导出] ./analysis/eciZM564_MM_reaction_kcat_MW_final_file.csv (2914 个反应)
[§3.iZM564_MM] 已生成 etcmodel: ./model/iZM564_MM_etcmodel.json (2915 个反应, 1641 个代谢物)


{'metabolites': [{'id': 'CARBOHYDRATES_c0',
   'name': 'CARBOHYDRATES',
   'compartment': 'c',
   'charge': 0,
   'annotation': {'sbo': 'SBO:0000247'},
   'concentration_ub': -3.912023005,
   'concentration_lb': -14.50865774},
  {'id': 'CARDIOLIPIN_c0',
   'name': 'CARDIOLIPIN',
   'compartment': 'c',
   'charge': 0,
   'annotation': {'sbo': 'SBO:0000247'},
   'concentration_ub': -3.912023005,
   'concentration_lb': -14.50865774},
  {'id': 'cpd00001_c0',
   'name': 'H2O',
   'compartment': 'c',
   'charge': 0,
   'formula': 'H2O',
   'annotation': {'bigg.metabolite': 'h2o; oh1',
    'biocyc': 'META:WATER',
    'cas': '7732-18-5',
    'chebi': 'CHEBI:15377',
    'chemspider': '937',
    'hmdb': 'HMDB02111',
    'inchikey': 'XLYOFNOQVPJJNP-UHFFFAOYSA-N',
    'kegg.compound': 'C00001',
    'metabolights': 'MTBLC15377',
    'metanetx.chemical': 'MNXM147877',
    'pubchem.compound': '962',
    'reactome': '5357732',
    'sbo': 'SBO:0000247',
    'seed.compound': 'cpd00001'},
   'concentrati

In [7]:
# §3.2 iZM564_MM015 (MM0.15 培养基)
build_etcmodel_variant("iZM564_MM015", CONFIG["variants"]["iZM564_MM015"], SHARED)

[导出] ./analysis/eciZM564_MM0.15_reaction_kcat_MW_final_file.csv (2914 个反应)
[§3.iZM564_MM015] 已生成 etcmodel: ./model/iZM564_MM015_etcmodel.json (2915 个反应, 1641 个代谢物)


{'metabolites': [{'id': 'CARBOHYDRATES_c0',
   'name': 'CARBOHYDRATES',
   'compartment': 'c',
   'charge': 0,
   'annotation': {'sbo': 'SBO:0000247'},
   'concentration_ub': -3.912023005,
   'concentration_lb': -14.50865774},
  {'id': 'CARDIOLIPIN_c0',
   'name': 'CARDIOLIPIN',
   'compartment': 'c',
   'charge': 0,
   'annotation': {'sbo': 'SBO:0000247'},
   'concentration_ub': -3.912023005,
   'concentration_lb': -14.50865774},
  {'id': 'cpd00001_c0',
   'name': 'H2O',
   'compartment': 'c',
   'charge': 0,
   'formula': 'H2O',
   'annotation': {'bigg.metabolite': 'h2o; oh1',
    'biocyc': 'META:WATER',
    'cas': '7732-18-5',
    'chebi': 'CHEBI:15377',
    'chemspider': '937',
    'hmdb': 'HMDB02111',
    'inchikey': 'XLYOFNOQVPJJNP-UHFFFAOYSA-N',
    'kegg.compound': 'C00001',
    'metabolights': 'MTBLC15377',
    'metanetx.chemical': 'MNXM147877',
    'pubchem.compound': '962',
    'reactome': '5357732',
    'sbo': 'SBO:0000247',
    'seed.compound': 'cpd00001'},
   'concentrati

## §4 kcat 参数迭代优化 (MM / MM015)

> MM 与 MM015 变体需要额外校准: 以实验实测生长速率为目标 (0.36 / 0.1533),
> 通过 `adjust_kcat_for_ETM` 迭代提升瓶颈反应的 kcat (上限来自 EC kcat 数据库
> `data/EC_kcat_max.json`), 输出 `ETiZM564_*_optimized.json`。
>
> 两变体的差异仅在目标生长速率,
> 均在 §1.2 的 `CONFIG["optimization"]` 中配置。
>
> 提示: 若需调整葡萄糖摄取范围 (如改为 (0, 75)), 直接修改 CONFIG 即可。

In [8]:
# §4.1 iZM564_MM: 目标生长速率 0.36, 葡萄糖固定 (65, 65)
result_mm, opt_mm = optimize_etm_kcat(
    etcmodel_path=CONFIG["variants"]["iZM564_MM"]["etcmodel_out"],
    ec_kcat_max_path=CONFIG["paths"]["ec_kcat_max"],
    **CONFIG["optimization"]["iZM564_MM"],
)

kcat 参数优化 - 初始求解
初始目标值 (biomass): 0.326750

--- 迭代 1/20 ---
  当前目标值: 0.326750
  总酶用量: 0.154275
  跳过 rxn01476: 数据库 kcat (14.30) <= 当前 kcat (14.30)
  当前目标值: 0.326750
  总酶用量: 0.154275
  (已跳过 2 个无法更新的瓶颈，继续尝试...)
  瓶颈: rxn05348 | 通量=0.0199 | 酶占比=9.12% | kcat=0.04 /h
  ✓ 更新 rxn05348: kcat 0.04 → 551.00 /h (EC: ['2.3.1.179', '2.3.1.41'])

--- 迭代 2/20 ---
  ✓ 目标达成: final_obj=0.360741, 差值 0.000741 <= 0.033250

kcat 参数优化 - 完成
初始目标值:     0.326750
最终目标值:     0.360741
相对提升:       10.40%
调整的反应数:   1
实际迭代次数:   1
模型已保存:     ./model/ETiZM564_MM_optimized.json
ETM optimize : 0.36074088949895194


In [9]:
# §4.2 iZM564_MM015: 目标生长速率 0.1533, 葡萄糖上限 (0, 65)
result_mm015, opt_mm015 = optimize_etm_kcat(
    etcmodel_path=CONFIG["variants"]["iZM564_MM015"]["etcmodel_out"],
    ec_kcat_max_path=CONFIG["paths"]["ec_kcat_max"],
    **CONFIG["optimization"]["iZM564_MM015"],
)

kcat 参数优化 - 初始求解
初始目标值 (biomass): 0.129810

--- 迭代 1/20 ---
  当前目标值: 0.129810
  总酶用量: 0.154275
  瓶颈: rxn00459 | 通量=21.2148 | 酶占比=18.00% | kcat=77.65 /h
  ✓ 更新 rxn00459: kcat 77.65 → 230.00 /h (EC: 4.2.1.11)

--- 迭代 2/20 ---
  当前目标值: 0.148509
  总酶用量: 0.154275
  瓶颈: rxn05347 | 通量=0.0299 | 酶占比=13.70% | kcat=0.04 /h
  ✓ 更新 rxn05347: kcat 0.04 → 551.00 /h (EC: ['2.3.1.179', '2.3.1.41'])

--- 迭代 3/20 ---
  ✓ 目标达成（回退）: final_obj=0.148509, 差值 0.004791 < 0.020333

kcat 参数优化 - 完成
初始目标值:     0.129810
最终目标值:     0.148509
相对提升:       14.41%
调整的反应数:   2
实际迭代次数:   2
模型已保存:     ./model/ETiZM564_MM015_optimized.json
ETM optimize : 0.14850921882339593
